# Etapa 1: Analise Exploratoria dos Dados e Diagnostico (NSL-KDD)

**Disciplina:** Inteligencia Artificial
**Entrega:** 24/10 | **Valor:** 2,00 pontos

Este notebook esta estruturado em 5 frentes de trabalho para divisao entre os 5 membros da equipe:
- **Membro 1:** Caracterizacao Geral da Base e Analise dos Tipos de Dados
- **Membro 2:** Analise das Classes de Ataque e Severidade do Desbalanceamento
- **Membro 3:** Analise dos Atributos Categoricos e Desafio da Alta Cardinalidade
- **Membro 4:** Deteccao de Redundancias, Atributos Constantes e Inconsistencias
- **Membro 5:** Investigacao de Vazamento de Dados (Data Leakage) e Diagnostico Consolidado

In [ ]:
import sys
from pathlib import Path

# Adiciona a raiz do projeto ao path para importar modulos de src
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import ALL_COLUMNS, CATEGORICAL_FEATURES, NUMERICAL_FEATURES, ATTACK_MAPPING, CLASSES_5
from src.data_loader import load_raw_dataframe

sns.set_theme(style="whitegrid")
print("Ambiente e bibliotecas carregados com sucesso.")

---
## Membro 1 — Caracterização geral da base e análise dos tipos de dados

**Objetivo:** caracterizar a estrutura e a completude do conjunto de treino do NSL-KDD, servindo de base comum para as análises dos demais membros.

**Escopo e cuidados metodológicos**
- Usamos **exclusivamente o `KDDTrain+`**, carregado com `load_raw_dataframe('train')`. O `KDDTest+` não é carregado nem consultado nesta seção, pois está reservado à avaliação final (seção 3 do enunciado). Assim, nenhuma informação do teste influencia o diagnóstico, evitando *data leakage*.
- Todas as colunas são preservadas, incluindo `label` e `difficulty_level`, porque esta etapa é diagnóstica.
- Aqui apenas **diagnosticamos e propomos**. A implementação e a comparação das técnicas de pré-processamento ficam para a Etapa 2.

### 1.1 Carregamento do conjunto de treino bruto

In [ ]:
# Opções de exibição usadas nesta seção
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# load_raw_dataframe (src.data_loader) já foi importado na célula de setup do notebook.
# Lê data/raw/KDDTrain+.txt com as 43 colunas nomeadas conforme src.config.ALL_COLUMNS.
df_train = load_raw_dataframe("train")
df_train.head()

### 1.2 Dimensões e papel de cada coluna

In [ ]:
COL_ROTULO = "label"
COL_METADADO = "difficulty_level"
COLS_ATRIBUTOS = [c for c in df_train.columns if c not in (COL_ROTULO, COL_METADADO)]

n_linhas, n_colunas = df_train.shape

# Verificações automáticas da estrutura esperada (41 atributos + label + difficulty_level)
assert n_colunas == 43, f"Esperadas 43 colunas, encontradas {n_colunas}"
assert len(COLS_ATRIBUTOS) == 41, f"Esperados 41 atributos, encontrados {len(COLS_ATRIBUTOS)}"

estrutura = pd.DataFrame({
    "Grupo": ["Atributos descritivos", "Rótulo (label)", "Metadado (difficulty_level)", "Total de colunas"],
    "Quantidade": [len(COLS_ATRIBUTOS), 1, 1, n_colunas],
})
print(f"Instâncias (linhas): {n_linhas:,}")
print(f"Colunas: {n_colunas}")
print(f"Uso de memória: {df_train.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
estrutura

In [ ]:
# Visão rápida das duas colunas que NÃO são atributos descritivos
print(f"label: tipo {df_train[COL_ROTULO].dtype}, {df_train[COL_ROTULO].nunique()} valores distintos "
      f"(normal + tipos de ataque)")
print(f"difficulty_level: tipo {df_train[COL_METADADO].dtype}, "
      f"intervalo [{df_train[COL_METADADO].min()}, {df_train[COL_METADADO].max()}], "
      f"{df_train[COL_METADADO].nunique()} valores distintos")
# difficulty_level é um metadado criado pelos autores da base (quantos classificadores de referência
# acertaram o registro). Ele NÃO é uma característica do tráfego e não deve ser usado como entrada
# do modelo; é mantido aqui apenas para o diagnóstico.

### 1.3 Mapeamento dos tipos dos atributos

Distinguimos dois níveis:
- **Tipo físico:** como o pandas armazenou a coluna (inteiro, real ou texto).
- **Tipo semântico:** o que o valor representa (categórico, binário, contagem inteira, taxa real). Isso importa porque um atributo armazenado como inteiro pode ser, na prática, uma variável binária ou categórica, e o tratamento na Etapa 2 depende dessa leitura.

In [ ]:
def tipo_fisico(serie: pd.Series) -> str:
    if pd.api.types.is_integer_dtype(serie):
        return "inteiro"
    if pd.api.types.is_float_dtype(serie):
        return "real"
    return "categórico (texto)"

def tipo_semantico(serie: pd.Series) -> str:
    # Regras derivadas dos próprios valores observados
    if not pd.api.types.is_numeric_dtype(serie):
        return "categórico nominal"
    valores = set(serie.unique())
    if len(valores) == 1:
        return "constante"
    if valores <= {0, 1}:
        return "binário (0/1)"
    if pd.api.types.is_float_dtype(serie) and serie.min() >= 0 and serie.max() <= 1:
        return "real – taxa em [0, 1]"
    if pd.api.types.is_integer_dtype(serie):
        return "inteiro – contagem/magnitude"
    return "real"

mapa_tipos = pd.DataFrame({
    "dtype_pandas": df_train[COLS_ATRIBUTOS].dtypes.astype(str),
    "tipo_fisico": [tipo_fisico(df_train[c]) for c in COLS_ATRIBUTOS],
    "tipo_semantico": [tipo_semantico(df_train[c]) for c in COLS_ATRIBUTOS],
    "n_valores_distintos": df_train[COLS_ATRIBUTOS].nunique(),
})
mapa_tipos

In [ ]:
# Resumo: quantos atributos em cada tipo físico e semântico
resumo_fisico = mapa_tipos["tipo_fisico"].value_counts().rename("qtd_atributos")
resumo_semantico = mapa_tipos["tipo_semantico"].value_counts().rename("qtd_atributos")
display(resumo_fisico.to_frame())
display(resumo_semantico.to_frame())

COLS_CATEGORICAS = mapa_tipos.index[mapa_tipos["tipo_fisico"] == "categórico (texto)"].tolist()
COLS_INTEIRAS = mapa_tipos.index[mapa_tipos["tipo_fisico"] == "inteiro"].tolist()
COLS_REAIS = mapa_tipos.index[mapa_tipos["tipo_fisico"] == "real"].tolist()
COLS_NUMERICAS = COLS_INTEIRAS + COLS_REAIS

print("Categóricos:", COLS_CATEGORICAS)
print(f"Inteiros ({len(COLS_INTEIRAS)}):", COLS_INTEIRAS)
print(f"Reais ({len(COLS_REAIS)}):", COLS_REAIS)

In [ ]:
# Atributos inteiros que, pela documentação do NSL-KDD, são binários (sim/não).
# Verificamos se os valores observados respeitam esse domínio.
BINARIOS_DOCUMENTADOS = ["land", "logged_in", "root_shell", "su_attempted",
                         "is_host_login", "is_guest_login"]

verif_binarios = pd.DataFrame({
    "valores_observados": [sorted(df_train[c].unique().tolist()) for c in BINARIOS_DOCUMENTADOS],
    "frequencia_valor_1": [(df_train[c] == 1).sum() for c in BINARIOS_DOCUMENTADOS],
    "dominio_0_1_respeitado": [set(df_train[c].unique()) <= {0, 1} for c in BINARIOS_DOCUMENTADOS],
}, index=BINARIOS_DOCUMENTADOS)
verif_binarios

In [ ]:
# Cardinalidade dos atributos categóricos (relevante para a escolha da codificação na Etapa 2)
card = pd.DataFrame({
    "n_categorias": df_train[COLS_CATEGORICAS].nunique(),
    "categoria_mais_frequente": [df_train[c].mode()[0] for c in COLS_CATEGORICAS],
    "pct_mais_frequente": [df_train[c].value_counts(normalize=True).iloc[0] * 100 for c in COLS_CATEGORICAS],
    "categorias_com_<10_registros": [(df_train[c].value_counts() < 10).sum() for c in COLS_CATEGORICAS],
})
card

### 1.4 Valores ausentes

In [ ]:
# (a) Ausentes explícitos (NaN/None) por coluna, em todas as 43 colunas
ausentes = pd.DataFrame({
    "n_ausentes": df_train.isna().sum(),
    "pct_ausentes": df_train.isna().mean() * 100,
})
print(f"Colunas com pelo menos um ausente: {(ausentes['n_ausentes'] > 0).sum()} de {n_colunas}")
print(f"Total de células ausentes: {int(ausentes['n_ausentes'].sum()):,} "
      f"de {df_train.size:,} células ({ausentes['n_ausentes'].sum() / df_train.size * 100:.4f}%)")
print(f"Linhas com pelo menos um ausente: {df_train.isna().any(axis=1).sum():,}")
ausentes.T

In [ ]:
# (b) Ausentes "disfarçados": marcadores que não viram NaN na leitura do arquivo.
# Em bases de rede é comum usar '?', '-', strings vazias ou valores negativos como "sem informação".
MARCADORES_TEXTO = {"?", "-", "", "na", "n/a", "nan", "null", "none"}
colunas_texto = COLS_CATEGORICAS + [COL_ROTULO]

marcadores = {
    c: df_train[c].astype(str).str.strip().str.lower().isin(MARCADORES_TEXTO).sum()
    for c in colunas_texto
}
negativos = (df_train[COLS_NUMERICAS + [COL_METADADO]] < 0).sum()

print("Marcadores textuais de ausência nas colunas categóricas/rótulo:", marcadores)
print("Colunas numéricas com valores negativos:", int((negativos > 0).sum()))

### 1.5 Estatísticas descritivas dos atributos numéricos

Além de média, desvio padrão e quartis, incluímos o coeficiente de variação, a assimetria e a proporção de zeros, que ajudam a justificar as propostas de transformação da Etapa 2.

In [ ]:
desc = df_train[COLS_NUMERICAS].describe().T  # count, mean, std, min, 25%, 50%, 75%, max
desc["cv"] = desc["std"] / desc["mean"].replace(0, np.nan)        # dispersão relativa
desc["assimetria"] = df_train[COLS_NUMERICAS].skew()
desc["pct_zeros"] = (df_train[COLS_NUMERICAS] == 0).mean() * 100
desc.insert(0, "tipo", mapa_tipos.loc[COLS_NUMERICAS, "tipo_semantico"])
desc

In [ ]:
# Destaques objetivos extraídos da tabela acima
print("Atributos com desvio padrão zero (constantes):",
      desc.index[desc["std"] == 0].tolist())
print("\nAtributos com mediana = 0 e Q3 = 0 (mais de 75% de zeros):")
print(desc.index[(desc["50%"] == 0) & (desc["75%"] == 0)].tolist())
print("\nMaiores amplitudes (max - min):")
print((desc["max"] - desc["min"]).sort_values(ascending=False).head(5).apply(lambda v: f"{v:,.0f}"))
print("\nAtributos com |assimetria| > 10:")
print(desc.loc[desc["assimetria"].abs() > 10, "assimetria"].sort_values(ascending=False).round(1))

### 1.6 Observações estruturais para os demais membros

Verificação rápida de redundância e conflito de rótulos. Não tratamos nada aqui: os números servem de ponto de partida para quem analisar dados redundantes/inconsistentes.

In [ ]:
dup_total = df_train.duplicated().sum()
dup_sem_dificuldade = df_train.drop(columns=[COL_METADADO]).duplicated().sum()

# Registros com os mesmos 41 atributos, mas rótulos diferentes (inconsistência de rótulo)
grupos = df_train.groupby(COLS_ATRIBUTOS, dropna=False)[COL_ROTULO].nunique()
grupos_conflitantes = grupos[grupos > 1]
linhas_conflitantes = df_train.merge(
    grupos_conflitantes.reset_index()[COLS_ATRIBUTOS], on=COLS_ATRIBUTOS, how="inner"
)

print(f"Linhas duplicadas (43 colunas): {dup_total}")
print(f"Linhas duplicadas ignorando difficulty_level: {dup_sem_dificuldade}")
print(f"Grupos de atributos idênticos com rótulos diferentes: {len(grupos_conflitantes)} "
      f"({len(linhas_conflitantes)} linhas envolvidas)")
(linhas_conflitantes
 .sort_values(["protocol_type", "service", "flag", "src_bytes", COL_ROTULO])
 [["protocol_type", "service", "flag", "src_bytes", "dst_bytes", COL_ROTULO, COL_METADADO]]
 .head(12))

### 1.7 Síntese: dimensões, tipos e completude

**Dimensões.** O `KDDTrain+` tem **125.973 instâncias e 43 colunas**: 41 atributos descritivos, o rótulo `label` (23 valores distintos: `normal` e 22 tipos de ataque) e o metadado `difficulty_level` (inteiro de 0 a 21). O `difficulty_level` descreve a dificuldade de classificação atribuída pelos autores da base, não o tráfego em si; por isso deve ser **excluído das entradas do modelo** na Etapa 2, pois usá-lo seria uma forma de vazamento de informação sobre o próprio rótulo.

**Tipos.** Pelo armazenamento, os 41 atributos se dividem em **23 inteiros, 15 reais e 3 categóricos** (`protocol_type`, `service`, `flag`). A leitura semântica, porém, é mais fina:
- Os 15 reais são todos **taxas no intervalo [0, 1]** (ex.: `serror_rate`, `same_srv_rate`), já em escala comparável entre si.
- Entre os inteiros, 5 são **binários** (`land`, `logged_in`, `root_shell`, `is_host_login`, `is_guest_login`), 17 são **contagens/magnitudes** e 1 é **constante**.
- `su_attempted`, documentado como binário, assume os valores **{0, 1, 2}**. É uma inconsistência de domínio a ser tratada (por exemplo, mapear 2 → 1) e discutida por quem analisar dados inconsistentes.
- Os categóricos têm cardinalidades muito diferentes: `protocol_type` tem 3 categorias, `flag` tem 11 e `service` tem **70**, das quais 8 aparecem menos de 10 vezes. Isso antecipa a discussão da conversão simbólico-numérica: *one-hot* em `service` gera ~70 colunas esparsas, o que motiva comparar com alternativas (agrupar categorias raras, *frequency* ou *target encoding* ajustados apenas dentro de cada *fold* de treino).

**Completude.** **Não há valores ausentes**: 0 de 5.416.839 células, em todas as 43 colunas. Também não foram encontrados ausentes disfarçados (`?`, `-`, strings vazias) nem valores negativos. Portanto, **não há necessidade de imputação** no treino. Como precaução, o pipeline da Etapa 2 pode manter um imputador ajustado só no treino, caso surjam ausentes em dados futuros.

**Distribuições numéricas.** As estatísticas mostram atributos fortemente assimétricos e dominados por zeros:
- `num_outbound_cmds` tem desvio padrão zero (constante) e não traz informação; é candidato direto à remoção.
- 21 atributos têm mediana e Q3 iguais a zero, isto é, mais de 75% dos registros valem 0 (ex.: `duration` vale 0 em 92% dos casos).
- `src_bytes` e `dst_bytes` vão de 0 a cerca de 1,3–1,4 bilhão, com desvio padrão na casa dos milhões e assimetria acima de 190. Já as taxas estão limitadas a [0, 1] e `dst_host_count` a 255.

Essa diferença de escalas de várias ordens de grandeza justifica, na Etapa 2, testar uma transformação como `log1p` nos atributos de bytes e contagens, seguida de normalização (`StandardScaler` ou `MinMaxScaler`), **sempre ajustada apenas no treino** (dentro da validação cruzada) e aplicada depois ao teste.

**Observações para os demais membros.** Não há linhas duplicadas, nem considerando todas as colunas, nem ignorando `difficulty_level`, o que confirma a limpeza feita pelos autores do NSL-KDD. Porém existem **15 grupos de registros com os mesmos 41 atributos e rótulos diferentes (31 linhas)**. Dez desses grupos são tráfego ICMP. Em 8 grupos o conflito é só entre ataques de varredura (`ipsweep`, `nmap`, `portsweep`), e nos outros 7 um registro `normal` aparece com o mesmo vetor de atributos de um ataque (`satan`, `pod`, `rootkit` e, em 4 grupos de `tcp/finger`, `land`). Esses conflitos são o ponto de partida para a análise de dados inconsistentes.

---
## Membro 2: Analise das Classes de Ataque e Severidade do Desbalanceamento

**Objetivos do Membro 2:**
1. Inspecionar a coluna de rotulo original (`label`), identificando quantos e quais tipos especificos de ataque estao presentes no treino.
2. Aplicar o agrupamento dos ataques nas 5 superclasses (`normal`, `dos`, `probe`, `r2l`, `u2r`) utilizando o mapeamento definido em `src.config.ATTACK_MAPPING`.
3. Calcular a distribuicao absoluta e percentual de cada superclasse no conjunto de treino.
4. Construir graficos que evidenciem visualmente a disparidade de proporcao entre as classes.
5. Discutir o impacto do desbalanceamento severo (especialmente a escassez da classe `u2r`) e por que o F1-score macro e essencial para a competicao em vez da simples acuracia.

In [ ]:
# TODO Membro 2: Inspecao dos rotulos originais e aplicacao do mapeamento para 5 classes
# Implemente aqui o calculo das contagens e porcentagens das classes

In [ ]:
# TODO Membro 2: Visualizacao grafica do desbalanceamento de classes
# Implemente aqui o grafico de barras da distribuicao das 5 classes

### Sintese do Membro 2
*[Espaco para o Membro 2 redigir a discussao sobre o desbalanceamento e a justificativa do F1-score macro]*

---
## Membro 3: Analise dos Atributos Categoricos e Desafio da Alta Cardinalidade

**Objetivos do Membro 3:**
1. Identificar e isolar os atributos categoricos (`protocol_type`, `service` e `flag`).
2. Determinar a cardinalidade (numero de valores unicos) de cada atributo categorico.
3. Analisar em detalhe o atributo `service`, gerando a contagem dos servicos mais frequentes e dos servicos raros.
4. Verificar se ha presenca de categorias no conjunto de teste que nao existem no treino (categorias nao vistas).
5. Discutir as implicacoes da alta cardinalidade de `service` caso se utilize One-Hot Encoding puro versus tecnicas alternativas (Frequency Encoding, agrupamento de categorias raras).

In [ ]:
# TODO Membro 3: Analise da cardinalidade de protocol_type, flag e service
# Implemente aqui a listagem dos valores unicos e contagem por atributo

In [ ]:
# TODO Membro 3: Analise aprofundada da distribuicao e cauda longa do atributo service
# Implemente aqui a analise dos top-k servicos e categorias de baixa frequencia

### Sintese do Membro 3
*[Espaco para o Membro 3 redigir a discussao tecnica sobre a alta cardinalidade de service e as estrategias de codificacao]*

---
## Membro 4: Deteccao de Redundancias, Atributos Constantes e Inconsistencias

**Objetivos do Membro 4:**
1. Investigar a ocorrencia de registros duplicados exatos (linhas identicas completas) no conjunto de treino.
2. Identificar atributos com variancia zero (constantes em todas as linhas), como `num_outbound_cmds`.
3. Investigar se existem instancias com atributos descritivos identicos mas com rotulos de classe divergentes (inconsistencias nos dados).
4. Avaliar atributos com variancia quase zero ou valores excessivamente concentrados em zero.
5. Justificar formalmente quais colunas ou registros devem ser descartados ou tratados na etapa de pre-processamento.

In [ ]:
# TODO Membro 4: Deteccao de duplicatas exatas e duplicatas parciais de features
# Implemente aqui a verificacao de linhas repetidas e inconsistencias de rotulo

In [ ]:
# TODO Membro 4: Deteccao de atributos constantes (variancia zero)
# Implemente aqui a checagem de desvio padrao / valores unicos nas colunas numericas

### Sintese do Membro 4
*[Espaco para o Membro 4 redigir a justificativa das inconsistencias e atributos redundantes identificados]*

---
## Membro 5: Investigacao de Vazamento de Dados (Data Leakage) e Diagnostico Final

**Objetivos do Membro 5:**
1. Analisar criticamente a coluna `difficulty_level`, demonstrando sua relacao com a classe e justificando por que ela NAO deve ser incluida como feature descritiva nos modelos.
2. Avaliar correlacao linear (ou associacao estatistica) entre as features numericas e a classe, verificando atributos que apresentem associacao suspeita ou espuria.
3. Verificar a correlacao entre atributos para identificar multicolinearidade severa.
4. Consolidar os apontamentos dos 5 membros em uma secao conclusiva de Diagnostico Geral dos Dados para submissao da Etapa 1.

In [ ]:
# TODO Membro 5: Analise da coluna difficulty_level e justificativa contra data leakage
# Implemente aqui a inspecao de difficulty_level por classe e demonstracao do risco metodologico

In [ ]:
# TODO Membro 5: Analise de correlacao de atributos com a classe e multicolinearidade
# Implemente aqui a matriz de correlacao ou analise de correlacao com o target

### Sintese do Membro 5 e Diagnostico Consolidado da Etapa 1
*[Espaco para o Membro 5 redigir a conclusao sobre prevencao de data leakage e consolidar o diagnostico geral da equipe para entrega]*